In [ ]:
nm_config = prepare_microsplit_project(
    data,
    data_type="array",
    axes="SYXC",
    n_channels=2,
    #everything else is default unless someone really wants to do something wacky
    
)

In [ ]:
def prepare_microsplit_project(
    ...
) -> NMConfig:

    default_n2v_config = create_n2v_configuration(
        experiment_name=f"blabla_n2v",
        data_type="array",
        axes="SYXC",
        patch_size=(64, 64),
        batch_size=64,
        num_epochs=5,
        n_channels=n_channels,
    )
    n2v_cfg = n2v_config or default_n2v_config

    n2v_runner = CAREamist(source=n2v_cfg, work_dir=n2v_work_dir)
    n2v_runner.train(
        train_source=n2v_train_source,
    )

    predictions = n2v_runner.predict()

    nm_paths: list[str] = []
    for channel_idx in range(n_channels):
        channel_signal = nm_input[..., channel_idx]
        channel_prediction = prediction_stack[:, channel_idx, ...]
        nm_config = GaussianMixtureNMConfig(
            min_signal=float(channel_signal.min()),
            max_signal=float(channel_signal.max()),
            **base_nm_kwargs,
        )
        noise_model = GaussianMixtureNoiseModel(nm_config)
        noise_model.fit(
            signal=channel_signal,
            observation=channel_prediction,
            n_epochs=noise_model_epochs,
        )
        model_name = f"{microsplit_experiment_name}_noise_model_ch{channel_idx}"
        noise_model.save(path=noise_model_path.as_posix(), name=model_name)
        nm_paths.append(str(noise_model_path / f"{model_name}.npz"))

    # optional here
    config = create_microsplit_configuration(
        experiment_name=microsplit_experiment_name,
        nm_paths=nm_paths,
        **microsplit_config_kwargs,
    )
    return config

In [ ]:
config = create_microsplit_configuration(
        experiment_name=microsplit_experiment_name,
        nm_config,
        ...
    )

In [ ]:
careamist = CAREamist(source=config)
careamist.train()
careamist.predict()